<a href="https://colab.research.google.com/github/RodrigoSampaio4/IA/blob/main/Projeto_Sistema_Especialista_Gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Projeto - Sistema Especialista com Integração Gemini

## Integrantes
- Rodrigo Sampaio - 062220027
- Rodrigo Batalha - 062220003
- Felipe Marques - 062220025
- Gabriel Damazio - 062220005

---

## Objetivo
Implementar um **Sistema Especialista** para análise de risco em equipamentos industriais com base em **temperatura**, **vibração** e **ruído**.  
A decisão técnica será tomada por regras rígidas do tipo **SE/ENTÃO**, enquanto a **API do Gemini** será usada apenas para gerar uma **análise interpretativa em linguagem natural**.

> **Importante:** não use no notebook a chave antiga que foi exposta. Gere uma **nova chave** no Google AI Studio antes de executar.

## Arquitetura Lógica

### Variáveis analisadas
- Temperatura
- Vibração
- Ruído

### Base de conhecimento
- **R1:** SE temperatura > 80 E vibração > 70 → **ALTO RISCO DE FALHA**
- **R2:** SE temperatura > 60 OU vibração > 50 → **RISCO MODERADO**
- **R3:** SE ruído > 70 → **VERIFICAR COMPONENTES**
- **R4:** SENÃO → **OPERAÇÃO NORMAL**

### Árvore de decisão
```text
Início
 ├─ Temperatura > 80 E Vibração > 70 ?
 │   └─ Sim → ALTO RISCO DE FALHA
 ├─ Temperatura > 60 OU Vibração > 50 ?
 │   └─ Sim → RISCO MODERADO
 ├─ Ruído > 70 ?
 │   └─ Sim → VERIFICAR COMPONENTES
 └─ Caso contrário → OPERAÇÃO NORMAL
```

## 1. Instalação da biblioteca necessária

In [1]:
!pip install -q -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 783.6/783.6 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.6/240.6 kB 14.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.49.2 which is incompatible.


## 2. Imports

In [2]:
import os
from getpass import getpass
from google import genai

## 3. Configuração da API Gemini

Cole **uma nova chave válida** quando for solicitado.

In [3]:
api_key = getpass('Digite sua nova GEMINI_API_KEY: ')
os.environ['GEMINI_API_KEY'] = api_key
client = genai.Client()

print('Cliente Gemini configurado com sucesso.')

Digite sua nova GEMINI_API_KEY: ··········
Cliente Gemini configurado com sucesso.


## 4. Base de conhecimento

In [4]:
BASE_DE_CONHECIMENTO = [
    {
        'id': 'R1',
        'descricao': 'Se temperatura > 80 e vibração > 70, então alto risco de falha.',
        'condicao': lambda temperatura, vibracao, ruido: temperatura > 80 and vibracao > 70,
        'decisao': 'ALTO RISCO DE FALHA',
        'justificativa_tecnica': 'Temperatura e vibração estão acima do limite crítico simultaneamente.'
    },
    {
        'id': 'R2',
        'descricao': 'Se temperatura > 60 ou vibração > 50, então risco moderado.',
        'condicao': lambda temperatura, vibracao, ruido: temperatura > 60 or vibracao > 50,
        'decisao': 'RISCO MODERADO',
        'justificativa_tecnica': 'Pelo menos uma variável ultrapassou o limite de atenção.'
    },
    {
        'id': 'R3',
        'descricao': 'Se ruído > 70, então verificar componentes.',
        'condicao': lambda temperatura, vibracao, ruido: ruido > 70,
        'decisao': 'VERIFICAR COMPONENTES',
        'justificativa_tecnica': 'Ruído elevado pode indicar desgaste, folga ou desalinhamento.'
    },
    {
        'id': 'R4',
        'descricao': 'Caso contrário, operação normal.',
        'condicao': lambda temperatura, vibracao, ruido: True,
        'decisao': 'OPERAÇÃO NORMAL',
        'justificativa_tecnica': 'Nenhuma condição crítica foi identificada.'
    }
]

print('Base de conhecimento carregada com sucesso.')

Base de conhecimento carregada com sucesso.


## 5. Motor de inferência do sistema especialista

In [5]:
def sistema_especialista(temperatura, vibracao, ruido):
    for regra in BASE_DE_CONHECIMENTO:
        if regra['condicao'](temperatura, vibracao, ruido):
            return {
                'temperatura': temperatura,
                'vibracao': vibracao,
                'ruido': ruido,
                'regra_acionada': regra['id'],
                'descricao_regra': regra['descricao'],
                'decisao': regra['decisao'],
                'justificativa_tecnica': regra['justificativa_tecnica']
            }

## 6. Função de análise interpretativa com Gemini

O Gemini **não toma a decisão técnica**. Ele apenas recebe a saída do sistema especialista e gera uma explicação, alertas e sugestões.

In [6]:
def analise_gemini(resultado):
    prompt = f"""
Você é um assistente técnico de apoio à manutenção industrial.

IMPORTANTE:
- Não altere a decisão técnica já tomada pelo sistema especialista.
- Sua função é apenas explicar o resultado em linguagem natural.
- Gere também alertas e sugestões estratégicas.

Dados:
Temperatura: {resultado['temperatura']}
Vibração: {resultado['vibracao']}
Ruído: {resultado['ruido']}

Saída do sistema especialista:
Regra acionada: {resultado['regra_acionada']}
Descrição da regra: {resultado['descricao_regra']}
Decisão técnica: {resultado['decisao']}
Justificativa técnica: {resultado['justificativa_tecnica']}

Responda em 3 partes:
1. Explicação da decisão
2. Alertas estratégicos
3. Sugestões de ajuste
"""

    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash-lite',
            contents=prompt
        )
        return response.text
    except Exception as e:
        return f'Erro na API Gemini: {e}'

## 7. Teste com um caso único

In [7]:
temperatura = 85
vibracao = 75
ruido = 40

resultado = sistema_especialista(temperatura, vibracao, ruido)

print('Decisão do Sistema:', resultado['decisao'])
print('Regra acionada:', resultado['regra_acionada'])
print('Justificativa Técnica:', resultado['justificativa_tecnica'])

resposta_gemini = analise_gemini(resultado)
print('\nGemini:\n', resposta_gemini)

Decisão do Sistema: ALTO RISCO DE FALHA
Regra acionada: R1
Justificativa Técnica: Temperatura e vibração estão acima do limite crítico simultaneamente.

Gemini:
 Compreendido! Como seu assistente técnico de apoio à manutenção industrial, estou pronto para explicar a decisão do sistema especialista e fornecer as informações necessárias para a tomada de ação.

---

### 1. Explicação da Decisão

O sistema especialista identificou um **ALTO RISCO DE FALHA** com base nos dados coletados. A **Regra R1** foi ativada porque dois parâmetros críticos atingiram níveis elevados simultaneamente:

*   **Temperatura:** Está em **85**, que é superior ao limite de **80** definido na regra.
*   **Vibração:** Está em **75**, que é superior ao limite de **70** definido na regra.

A **justificativa técnica** é clara: a combinação de **temperatura elevada e vibração acima do limite crítico** indica uma situação de risco iminente, onde a probabilidade de uma falha no equipamento é significativamente alta. O 

## 8. Testes com múltiplos casos e geração de logs

In [8]:
casos_teste = [
    {'temperatura': 85, 'vibracao': 75, 'ruido': 40},
    {'temperatura': 65, 'vibracao': 30, 'ruido': 45},
    {'temperatura': 40, 'vibracao': 20, 'ruido': 80},
    {'temperatura': 35, 'vibracao': 25, 'ruido': 30},
]

for i, caso in enumerate(casos_teste, start=1):
    resultado = sistema_especialista(
        caso['temperatura'],
        caso['vibracao'],
        caso['ruido']
    )

    print('=' * 60)
    print(f'CASO {i}')
    print(f"Temperatura: {resultado['temperatura']}")
    print(f"Vibração: {resultado['vibracao']}")
    print(f"Ruído: {resultado['ruido']}")
    print(f"Regra acionada: {resultado['regra_acionada']}")
    print(f"Decisão do sistema: {resultado['decisao']}")
    print(f"Justificativa técnica: {resultado['justificativa_tecnica']}")

    resposta = analise_gemini(resultado)
    print('\nResposta do Gemini:')
    print(resposta)
    print()

CASO 1
Temperatura: 85
Vibração: 75
Ruído: 40
Regra acionada: R1
Decisão do sistema: ALTO RISCO DE FALHA
Justificativa técnica: Temperatura e vibração estão acima do limite crítico simultaneamente.

Resposta do Gemini:
Com certeza! Vamos analisar os dados e a decisão técnica.

### 1. Explicação da Decisão

A decisão técnica de **ALTO RISCO DE FALHA** foi tomada com base na ativação da Regra R1. Esta regra é projetada para identificar situações onde a combinação de parâmetros indica um perigo iminente para o equipamento.

No seu caso, os valores medidos foram:
*   **Temperatura: 85**
*   **Vibração: 75**

A Regra R1 considera que se a **Temperatura for superior a 80** E a **Vibração for superior a 70**, então o risco de falha é classificado como alto. Como ambos os seus valores (85 e 75) excedem os limites definidos na regra (80 e 70, respectivamente), a condição para "alto risco de falha" foi plenamente atendida. Isso significa que a combinação desses dois parâmetros indica um cenário 

## 9. Conclusão

Neste projeto foi utilizada a abordagem de **Sistema Especialista**, adequada para problemas de diagnóstico baseados em regras rígidas.  
A base de conhecimento foi construída com regras **SE/ENTÃO**, permitindo ao sistema classificar o estado do equipamento a partir dos valores de temperatura, vibração e ruído.  
Após a tomada de decisão técnica, a **API do Gemini** foi integrada para gerar uma **análise interpretativa**, explicando a decisão em linguagem natural e sugerindo alertas e ajustes estratégicos.

Esse fluxo atende ao enunciado porque separa claramente:
- **a decisão técnica**, feita pelo sistema especialista;
- **a interpretação textual**, feita pela IA generativa.